# qpl Pricing Overview

Compare analytic, Monte Carlo, and PDE pricing for one European option in a single workflow.

## Setup

This notebook uses deterministic seeds and smoke-mode gates for CI-scale runs.

In [ ]:
import matplotlib.pyplot as plt
from qpl.engines.mc.pricers import MCConfig
from qpl.engines.pde.pricers import PDEConfig
from qpl.instruments.options import EuropeanOption
from qpl.market.curves import FlatDividendCurve, FlatRateCurve
from qpl.market.market import Market
from qpl.models.black_scholes import BlackScholesModel
from qpl.pricing import price
from qpl.utils import choose_by_mode, is_smoke_mode, set_global_seed

SEED = 123
_ = set_global_seed(SEED)
SMOKE_MODE = is_smoke_mode()
plt.style.use("seaborn-v0_8-whitegrid")

MC_PATHS = choose_by_mode(SMOKE_MODE, smoke=3_000, full=30_000)
MC_STEPS = choose_by_mode(SMOKE_MODE, smoke=30, full=120)
PDE_GRID = choose_by_mode(SMOKE_MODE, smoke=50, full=160)

option = EuropeanOption(kind="call", strike=100.0, expiry=1.0)
model = BlackScholesModel(sigma=0.20)
market = Market(
    spot=100.0,
    rate_curve=FlatRateCurve(0.05),
    dividend_curve=FlatDividendCurve(0.01),
)

print(f"seed={SEED} smoke_mode={SMOKE_MODE} mc_paths={MC_PATHS} pde_grid={PDE_GRID}")

## Price Comparison

In [ ]:
analytic = price(option, model, market, method="analytic")
mc_cfg = MCConfig(n_paths=MC_PATHS, n_steps=MC_STEPS, seed=SEED)
mc = price(option, model, market, method="mc", cfg=mc_cfg)
pde_cfg = PDEConfig(n_s=PDE_GRID, n_t=PDE_GRID, theta=0.5, s_max_multiplier=4.0)
pde = price(option, model, market, method="pde", cfg=pde_cfg)

mc_stderr = mc.stderr or 0.0
print(f"analytic={analytic.value:.6f}")
print(f"mc={mc.value:.6f} stderr={mc_stderr:.6f}")
print(f"pde={pde.value:.6f}")
print(f"abs_err_mc={abs(mc.value - analytic.value):.6f}")
print(f"abs_err_pde={abs(pde.value - analytic.value):.6f}")

## Theta-Scheme Snapshot

In [ ]:
theta_values = [0.0, 0.5, 1.0]
theta_prices = []
for theta in theta_values:
    cfg = PDEConfig(n_s=PDE_GRID, n_t=PDE_GRID, theta=theta, s_max_multiplier=4.0)
    px = price(option, model, market, method="pde", cfg=cfg).value
    theta_prices.append(px)
    print(f"theta={theta:.1f} price={px:.6f} abs_err={abs(px - analytic.value):.6f}")

plt.figure(figsize=(6.5, 3.5))
plt.plot(theta_values, theta_prices, marker="o", label="PDE price")
plt.axhline(analytic.value, linestyle="--", color="k", label="analytic")
plt.xlabel("theta")
plt.ylabel("price")
plt.title("Theta-scheme sensitivity (single grid)")
plt.legend()
plt.show()

## Takeaways

- Analytic results provide a stable benchmark for numerical methods.
- MC uncertainty is read with its standard error, not price difference alone.
- PDE theta choice matters; `theta=0.5` is a strong default for this setup.